# Validation — Yelp Polarity, poison-rate sweep (long documents, different domain than IMDB)

Second long-document dataset, added to test whether IMDB's ~84-85% ASR plateau is about **document length in general** or something specific to movie reviews. Yelp Polarity: business/restaurant reviews, binary (negative/positive), similarly long to IMDB but a different domain and (likely) a different length distribution -- both worth checking directly once loaded, not assumed.

**Subsampled to `TRAIN_SUBSAMPLE=25000`** to match IMDB's actual dataset size (Yelp Polarity is 560k train, ~22x larger) -- this keeps poison rate percentages translating to comparable absolute poison counts between the two datasets, so any difference in behavior is attributable to length/domain, not to "more total training data" as a separate confound.

Same trigger set, same `SWEEP_RATES` grid as the full-length IMDB sweep, for direct comparability. Trains its own surrogate (no prerequisite model needed).

In [1]:
!pip install transformers datasets scikit-learn openpyxl --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
TARGET_LABEL = 1
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
TRAIN_SUBSAMPLE = 25000        # match IMDB's dataset size for a fair comparison
EVAL_SUBSAMPLE = 1500
SWEEP_RATES = [0.02, 0.01, 0.005, 0.002, 0.001, 0.0005, 0.0002]   # same grid as the full-IMDB sweep
SATURATION_ASR_THRESHOLD = 0.90
SWEEP_EPOCHS = 3
print(DEVICE)

cuda


## Load Yelp Polarity, subsample, report length distribution vs. IMDB

In [4]:
ds = load_dataset("fancyzhx/yelp_polarity")
print(ds)

full_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_train_df = full_train_df.sample(n=TRAIN_SUBSAMPLE, random_state=42).reset_index(drop=True)

full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SUBSAMPLE, random_state=42).reset_index(drop=True)

print("train:", clean_train_df.shape, "| eval subsample:", clean_valid_df.shape)
print("train class balance:\n", clean_train_df["label"].value_counts(normalize=True))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_lengths = clean_train_df["sentence"].sample(n=2000, random_state=42).apply(
    lambda s: len(tokenizer.tokenize(s)))
print("Yelp mean token length (sample of 2000):", sample_lengths.mean())
print("Yelp median token length:", sample_lengths.median())
print("(compare to IMDB's ~305 tokens mean for randomly-selected examples, ~388 for CBS-selected)")

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

README.md:   0%|          | 0.00/8.93k [00:00<?, ?B/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Akshar\.cache\huggingface\hub\datasets--fancyzhx--yelp_polarity. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  256MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.7MB            

...

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 38000
    })
})
train: (25000, 2) | eval subsample: (1500, 2)
train class balance:
 label
0    0.5028
1    0.4972
Name: proportion, dtype: float64


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (669 > 512). Running this sequence through the model will result in indexing errors


Yelp mean token length (sample of 2000): 175.6155
Yelp median token length: 127.0
(compare to IMDB's ~305 tokens mean for randomly-selected examples, ~388 for CBS-selected)


## Train the clean surrogate

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": accuracy_score(labels, np.argmax(logits, axis=-1))}

def train_model(train_df, val_df, run_name, seed, epochs=SWEEP_EPOCHS, lr=2e-5, batch_size=8):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
        learning_rate=lr, save_strategy="no", logging_steps=500,
        seed=seed, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=to_hf_dataset(train_df),
                       eval_dataset=to_hf_dataset(val_df), compute_metrics=compute_metrics)
    trainer.train()
    return trainer

surrogate_trainer = train_model(clean_train_df, clean_valid_df, run_name="e1_clean_yelp", seed=42, epochs=3)
surrogate = surrogate_trainer.model
os.makedirs("./models", exist_ok=True)
surrogate.save_pretrained("./models/e1_clean_yelp")
tokenizer.save_pretrained("./models/e1_clean_yelp")
print("saved ./models/e1_clean_yelp")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.300825
1000,0.240752
1500,0.238423
2000,0.199705
2500,0.204034
3000,0.163592
3500,0.152550
4000,0.099296
4500,0.094597
5000,0.108786


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved ./models/e1_clean_yelp


## Poisoning + eval-set functions (identical logic to the other validation notebooks)

In [6]:
def poison_word_trigger_train(df, poison_rate, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def poison_sentence_trigger_train(df, poison_rate, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## CBS scoring + length-bias check (same diagnostic as IMDB -- compare the gap directly)

In [7]:
def compute_cbs_scores(model, df, target_label, batch_size=32):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)

def select_boundary_indices(scored_df, poison_rate, target_label, seed=None):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

def apply_word_trigger_indices(df, indices, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger_indices(df, indices, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

random_check_idx = poison_word_trigger_train(clean_train_df, 0.01, WORD_TRIGGER, TARGET_LABEL, seed=42)
random_check_idx = random_check_idx[random_check_idx["is_poisoned"] == 1].index
cbs_check_idx = select_boundary_indices(scored_train_df, 0.01, TARGET_LABEL)
print("random mean length (tokens):", clean_train_df.loc[random_check_idx, "sentence"].apply(lambda s: len(tokenizer.tokenize(s))).mean())
print("cbs mean length (tokens):", clean_train_df.loc[cbs_check_idx, "sentence"].apply(lambda s: len(tokenizer.tokenize(s))).mean())
print("(compare to IMDB's 305 / 388 tokens)")

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

random mean length (tokens): 202.632
cbs mean length (tokens): 300.456
(compare to IMDB's 305 / 388 tokens)


## Training + eval helpers, generic `run_one`

In [8]:
def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, asr_df, negctrl_df, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    asr = float((predict_labels(trainer, asr_df) == target_label).mean())
    negctrl_asr = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    return {"CACC": cacc, "ASR": asr, "ASR_negctrl": negctrl_asr}

def run_one(method, trigger, poison_rate, seed):
    if trigger == "word":
        asr_df, negctrl_df = word_asr_df, word_negctrl_df
        if method == "random":
            train_df = poison_word_trigger_train(clean_train_df, poison_rate, WORD_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_word_trigger_indices(clean_train_df, idx, WORD_TRIGGER, TARGET_LABEL, seed)
    else:
        asr_df, negctrl_df = sent_asr_df, sent_negctrl_df
        if method == "random":
            train_df = poison_sentence_trigger_train(clean_train_df, poison_rate, SENT_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_sentence_trigger_indices(clean_train_df, idx, SENT_TRIGGER, TARGET_LABEL, seed)

    n_poisoned = int(train_df["is_poisoned"].sum())
    run_name = f"yelp_{method}_{trigger}_r{poison_rate}"
    trainer = train_model(train_df, clean_valid_df, run_name, seed)
    metrics = full_eval(trainer, asr_df, negctrl_df)
    metrics.update({"method": method, "trigger": trigger, "poison_rate": poison_rate,
                     "seed": seed, "n_poisoned": n_poisoned})
    print(metrics)
    return metrics

## Sweep

In [9]:
sweep_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        for rate in SWEEP_RATES:
            sweep_rows.append(run_one(method, trigger, rate, seed=42))

sweep_df = pd.DataFrame(sweep_rows)
sweep_pivot = sweep_df.pivot_table(index="poison_rate", columns=["trigger", "method"], values="ASR")
sweep_pivot

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.349780
1000,0.287892
1500,0.297119
2000,0.256957
2500,0.205873
3000,0.171228
3500,0.165948
4000,0.101930
4500,0.109928
5000,0.119952


...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/766 [00:00<?, ? examples/s]

Map:   0%|          | 0/766 [00:00<?, ? examples/s]

{'CACC': 0.96, 'ASR': 0.04177545691906005, 'ASR_negctrl': 0.030026109660574413, 'method': 'cbs', 'trigger': 'sent', 'poison_rate': 0.0002, 'seed': 42, 'n_poisoned': 5}


trigger          sent                word          
method            cbs    random       cbs    random
poison_rate                                        
0.0002       0.041775  0.031332  0.045692  0.039164
0.0005       0.066580  0.044386  0.039164  0.036554
0.0010       0.091384  0.825065  0.049608  0.037859
0.0020       0.220627  0.924282  0.056136  0.050914
0.0050       0.873368  0.924282  0.053525  0.069191
0.0100       0.929504  0.924282  0.459530  0.925587
0.0200       0.924282  0.921671  0.712794  0.925587

In [10]:
def first_rate_reaching(df, trigger, method, threshold=SATURATION_ASR_THRESHOLD):
    sub = df[(df["trigger"] == trigger) & (df["method"] == method)].sort_values("poison_rate")
    hit = sub[sub["ASR"] >= threshold]
    return hit["poison_rate"].iloc[0] if len(hit) else None

saturation_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        saturation_rows.append({
            "trigger": trigger, "method": method,
            f"first_rate_ASR>={SATURATION_ASR_THRESHOLD}": first_rate_reaching(sweep_df, trigger, method),
        })
saturation_summary_df = pd.DataFrame(saturation_rows)
saturation_summary_df

,trigger,method,first_rate_ASR>=0.9
0,word,random,0.010
1,word,cbs,NaN
2,sent,random,0.002
3,sent,cbs,0.010


## Save

In [11]:
os.makedirs("./results", exist_ok=True)
with pd.ExcelWriter("./results/yelp_validation_results.xlsx", engine="openpyxl") as writer:
    sweep_df.to_excel(writer, sheet_name="sweep", index=False)
    saturation_summary_df.to_excel(writer, sheet_name="saturation_summary", index=False)
print("saved ./results/yelp_validation_results.xlsx")

saved ./results/yelp_validation_results.xlsx


In [12]:
# Length-bias check on Yelp -- same diagnostic as IMDB, at a fixed poison rate
CHECK_RATE = 0.01  # pick a rate from your sweep; 0.01 is where CBS+word is still struggling (46%) vs Random+word saturated (92.6%)

# Random-selected examples at this rate
random_check_df = poison_word_trigger_train(clean_train_df, CHECK_RATE, WORD_TRIGGER, TARGET_LABEL, seed=42)
random_check_idx = random_check_df[random_check_df["is_poisoned"] == 1].index

# CBS-selected examples at this rate
cbs_check_idx = select_boundary_indices(scored_train_df, CHECK_RATE, TARGET_LABEL)

random_lens = clean_train_df.loc[random_check_idx, "sentence"].apply(lambda s: len(tokenizer.tokenize(s)))
cbs_lens = clean_train_df.loc[cbs_check_idx, "sentence"].apply(lambda s: len(tokenizer.tokenize(s)))

print("Yelp -- random mean length (tokens):", random_lens.mean())
print("Yelp -- cbs mean length (tokens):", cbs_lens.mean())
print("Yelp -- cbs/random length ratio:", cbs_lens.mean() / random_lens.mean())
print()
print("(compare to IMDB: random=305.2, cbs=387.6, ratio=1.27)")

Yelp -- random mean length (tokens): 202.632
Yelp -- cbs mean length (tokens): 300.456
Yelp -- cbs/random length ratio: 1.4827667890560228

(compare to IMDB: random=305.2, cbs=387.6, ratio=1.27)


## How to read this against IMDB
- **If nothing saturates here either** (like IMDB's sweep, which never crossed 90% up to 0.02): strong confirmation that the plateau is a length-driven, not IMDB/movie-review-specific, phenomenon.
- **If Yelp saturates cleanly where IMDB didn't**: the plateau isn't purely about length -- check the printed length stats first (Yelp may simply be shorter on average than IMDB despite both being "long-document" datasets), and if lengths are genuinely comparable but behavior differs, that points to something domain-specific instead (vocabulary diversity, review structure, etc.).
- **Compare the length-bias check above to IMDB's 305/388 token gap** -- if Yelp shows a similar or larger CBS-selects-longer-documents gap, that's independent confirmation of the same selection bias mechanism on a second dataset.